In [22]:
import torch
from collections import Counter
from domain.configs import MAX_STEPS_PER_EPISODE
from environment.canonical_version.grenight_environment import GrenightEnvironment
from agents.double_dqn_self_play.agent import Agent

In [23]:
def play_game(env_arg: GrenightEnvironment,
              agent_arg: Agent,
              is_agent_playing_for_white: bool,
              is_agent_playing_for_black: bool,
              second_agent: Agent | None=None) -> tuple[str, dict, int]:

    if is_agent_playing_for_white and is_agent_playing_for_black:
        eps = 0.05
    else:
        eps = 0.0

    state = env_arg.reset()
    done = False
    move_count = 0
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        if acting_player_is_white:
            if is_agent_playing_for_white:
                action = agent_arg.select_action(state, env_arg.action_mask(), eps)
            else:
                action = env_arg.sample()
        else:
            if is_agent_playing_for_black:
                if second_agent is None:
                    action = agent_arg.select_action(state, env_arg.action_mask(), eps)
                else:
                    action = second_agent.select_action(state, env_arg.action_mask(), eps)
            else:
                action = env_arg.sample()

        state, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", info, move_count
    if reward == 0.0:
        return "draw", info, move_count

    return ("white_win", info, move_count) if reward == 1.0 else ("black_win", info, move_count)

#### Evaluating self-play through checkpoints, one checkpoint playing against all checkpoints for both colors with eps=0.05 for both colors

In [25]:
device = "cpu"

for i in range(1, 6):
    for j in range(1, i):
        env = GrenightEnvironment()
        outcomes_counter = Counter()
        draw_reasons_counter = Counter()

        agent_white = Agent(
            num_planes=env.state_encoder.NUM_PLANES,
            rows=5,
            columns=4,
            num_actions=env.action_encoder.NUM_ACTIONS,
            device=device
        )

        checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

        agent_white.policy_net.load_state_dict(checkpoint["policy_state_dict"])
        agent_white.target_net.load_state_dict(checkpoint["target_state_dict"])
        agent_white.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        agent_black = Agent(
            num_planes=env.state_encoder.NUM_PLANES,
            rows=5,
            columns=4,
            num_actions=env.action_encoder.NUM_ACTIONS,
            device=device
        )

        checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{j * 5_000}.pt", map_location=device, weights_only=False)

        agent_black.policy_net.load_state_dict(checkpoint["policy_state_dict"])
        agent_black.target_net.load_state_dict(checkpoint["target_state_dict"])
        agent_black.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        for _ in range(1_000):
            outcome, game_info, move_count = play_game(env, agent_white, True, True, agent_black)
            outcomes_counter[outcome] += 1
            if game_info["draw_reason"] is not None:
                draw_reasons_counter[game_info["draw_reason"]] += 1

        print(f"STATS OUT FROM: {1_000} GAMES AT: white={i * 5_000} VS black={j * 5_000} CHECKPOINT\n"
              f"Outcomes: {outcomes_counter}\n"
              f"Draw reasons: {draw_reasons_counter}\n")

        env = GrenightEnvironment()
        outcomes_counter = Counter()
        draw_reasons_counter = Counter()

        agent_black = Agent(
            num_planes=env.state_encoder.NUM_PLANES,
            rows=5,
            columns=4,
            num_actions=env.action_encoder.NUM_ACTIONS,
            device=device
        )

        checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

        agent_black.policy_net.load_state_dict(checkpoint["policy_state_dict"])
        agent_black.target_net.load_state_dict(checkpoint["target_state_dict"])
        agent_black.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        agent_white = Agent(
            num_planes=env.state_encoder.NUM_PLANES,
            rows=5,
            columns=4,
            num_actions=env.action_encoder.NUM_ACTIONS,
            device=device
        )

        checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{j * 5_000}.pt", map_location=device, weights_only=False)

        agent_white.policy_net.load_state_dict(checkpoint["policy_state_dict"])
        agent_white.target_net.load_state_dict(checkpoint["target_state_dict"])
        agent_white.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        for _ in range(1_000):
            outcome, game_info, move_count = play_game(env, agent_white, True, True, agent_black)
            outcomes_counter[outcome] += 1
            if game_info["draw_reason"] is not None:
                draw_reasons_counter[game_info["draw_reason"]] += 1

        print(f"STATS OUT FROM: {1_000} GAMES AT: white={j * 5_000} VS black={i * 5_000} CHECKPOINT\n"
              f"Outcomes: {outcomes_counter}\n"
              f"Draw reasons: {draw_reasons_counter}\n")

    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])


    for _ in range(1_000):
        outcome, game_info, move_count = play_game(env, agent, True, True)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1

    print(f"STATS OUT FROM: {1_000} SELF-PLAY GAMES AT: {i * 5_000} CHECKPOINT\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 SELF-PLAY GAMES AT: 5000 CHECKPOINT
Outcomes: Counter({'draw': 673, 'black_win': 214, 'white_win': 113})
Draw reasons: Counter({'insufficient_material': 481, 'threefold_repetition': 122, 'stalemate': 70})

STATS OUT FROM: 1000 GAMES AT: white=10000 VS black=5000 CHECKPOINT
Outcomes: Counter({'white_win': 717, 'draw': 155, 'black_win': 128})
Draw reasons: Counter({'threefold_repetition': 76, 'insufficient_material': 57, 'stalemate': 22})

STATS OUT FROM: 1000 GAMES AT: white=5000 VS black=10000 CHECKPOINT
Outcomes: Counter({'draw': 665, 'black_win': 173, 'white_win': 162})
Draw reasons: Counter({'insufficient_material': 437, 'threefold_repetition': 167, 'stalemate': 51, 'max_steps_without_progress': 10})

STATS OUT FROM: 1000 SELF-PLAY GAMES AT: 10000 CHECKPOINT
Outcomes: Counter({'white_win': 716, 'draw': 187, 'black_win': 97})
Draw reasons: Counter({'threefold_repetition': 111, 'insufficient_material': 39, 'stalemate': 34, 'max_steps_without_progress': 3})

STATS 

#### Evaluating self-play through checkpoints, agent picking for white at current checkpoint, random valid actions for black with eps=0.0 for white

In [27]:
device = "cpu"

for i in range(1, 6):
    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for _ in range(1_000):
        outcome, game_info, move_count = play_game(env, agent, True, False)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1

    print(f"STATS OUT FROM: {1_000} GAMES AT: white={i * 5_000} CHECKPOINT VS black=RANDOM\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 GAMES AT: white=5000 CHECKPOINT VS black=RANDOM
Outcomes: Counter({'draw': 494, 'white_win': 394, 'black_win': 112})
Draw reasons: Counter({'insufficient_material': 190, 'threefold_repetition': 150, 'stalemate': 149, 'max_steps_without_progress': 5})

STATS OUT FROM: 1000 GAMES AT: white=10000 CHECKPOINT VS black=RANDOM
Outcomes: Counter({'white_win': 438, 'draw': 427, 'black_win': 135})
Draw reasons: Counter({'insufficient_material': 154, 'threefold_repetition': 134, 'stalemate': 119, 'max_steps_without_progress': 20})

STATS OUT FROM: 1000 GAMES AT: white=15000 CHECKPOINT VS black=RANDOM
Outcomes: Counter({'white_win': 528, 'draw': 393, 'black_win': 79})
Draw reasons: Counter({'insufficient_material': 150, 'threefold_repetition': 119, 'stalemate': 111, 'max_steps_without_progress': 13})

STATS OUT FROM: 1000 GAMES AT: white=20000 CHECKPOINT VS black=RANDOM
Outcomes: Counter({'white_win': 577, 'draw': 349, 'black_win': 74})
Draw reasons: Counter({'insufficient_mat

#### Evaluating self-play through checkpoints, agent picking for black at current checkpoint, random valid actions for white with eps=0.0 for black

In [28]:
device = "cpu"

for i in range(1, 6):
    env = GrenightEnvironment()
    outcomes_counter = Counter()
    draw_reasons_counter = Counter()

    agent = Agent(
        num_planes=env.state_encoder.NUM_PLANES,
        rows=5,
        columns=4,
        num_actions=env.action_encoder.NUM_ACTIONS,
        device=device
    )

    checkpoint = torch.load(f"../double_dqn_self_play/checkpoints/ep{i * 5_000}.pt", map_location=device, weights_only=False)

    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for _ in range(1_000):
        outcome, game_info, move_count = play_game(env, agent, False, True)
        outcomes_counter[outcome] += 1
        if game_info["draw_reason"] is not None:
            draw_reasons_counter[game_info["draw_reason"]] += 1

    print(f"STATS OUT FROM: {1_000} GAMES AT: white=RANDOM VS white={i * 5_000} CHECKPOINT\n"
          f"Outcomes: {outcomes_counter}\n"
          f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 GAMES AT: white=RANDOM VS white=5000 CHECKPOINT
Outcomes: Counter({'draw': 454, 'black_win': 439, 'white_win': 107})
Draw reasons: Counter({'threefold_repetition': 172, 'insufficient_material': 147, 'stalemate': 124, 'max_steps_without_progress': 11})

STATS OUT FROM: 1000 GAMES AT: white=RANDOM VS white=10000 CHECKPOINT
Outcomes: Counter({'black_win': 471, 'draw': 418, 'white_win': 111})
Draw reasons: Counter({'threefold_repetition': 154, 'insufficient_material': 153, 'stalemate': 98, 'max_steps_without_progress': 13})

STATS OUT FROM: 1000 GAMES AT: white=RANDOM VS white=15000 CHECKPOINT
Outcomes: Counter({'black_win': 488, 'draw': 422, 'white_win': 90})
Draw reasons: Counter({'insufficient_material': 167, 'threefold_repetition': 143, 'stalemate': 95, 'max_steps_without_progress': 17})

STATS OUT FROM: 1000 GAMES AT: white=RANDOM VS white=20000 CHECKPOINT
Outcomes: Counter({'black_win': 537, 'draw': 379, 'white_win': 84})
Draw reasons: Counter({'insufficient_mate